# Huấn luyện Mô hình Học máy Hiệu chỉnh Dự báo Mưa (Trạm Bản Hồ)

## 1. Mục tiêu bài toán & Cải tổ Dữ liệu
Các mô hình dự báo thời tiết số trị (**ECMWF**, **GFS**, **Meteoblue**) đưa ra dự báo lượng mưa cho từng giờ trong tương lai. Mục tiêu của notebook này là xây dựng mô hình Học máy (**Machine Learning Post-Processing / MOS**) kết hợp cả 3 nguồn dự báo trên **16 đặc trưng cốt lõi** để **ước lượng chính xác lượng mưa thực đo (`obs`)** bậc `0.2 mm`.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor,
    HistGradientBoostingClassifier
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    f1_score,
    accuracy_score,
    confusion_matrix
)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
print("Đã nạp thành công các thư viện!")

Đã nạp thành công các thư viện!


## 2. Đọc dữ liệu Đã Cải tổ (10 cột) và Khám phá dữ liệu (EDA)
Đã loại bỏ 5 cột dư thừa/gây học vẹt trong file CSV (`nam`, `ngay`, `ecmwf_phancap`, `gfs_phancap`, `mblue_phancap`), chỉ giữ lại 10 cột cốt lõi: `thoigian`, `thang`, `gio`, `ecmwf_1h`, `gfs_1h`, `mblue_1h`, `ecmwf_24h`, `gfs_24h`, `mblue_24h`, `obs`.

In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name in ("notebooks", "tests", "src") else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "du_lieu_banho.csv"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
cols_10 = ['thoigian', 'thang', 'gio', 'ecmwf_1h', 'gfs_1h', 'mblue_1h', 'ecmwf_24h', 'gfs_24h', 'mblue_24h', 'obs']
df = pd.read_csv(DATA_PATH)[cols_10]
df['thoigian'] = pd.to_datetime(df['thoigian'])
df = df.sort_values('thoigian').reset_index(drop=True)

print(f"Kích thước dữ liệu: {df.shape}")
print(f"Khoảng thời gian : {df['thoigian'].min()} -> {df['thoigian'].max()}")
display(df.head())

Kích thước dữ liệu: (21376, 10)
Khoảng thời gian : 2024-01-01 00:00:00 -> 2026-06-09 15:00:00


,thoigian,thang,gio,ecmwf_1h,gfs_1h,mblue_1h,ecmwf_24h,gfs_24h,mblue_24h,obs
0,2024-01-01 00:00:00,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2024-01-01 01:00:00,1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2024-01-01 02:00:00,1,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2024-01-01 03:00:00,1,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2024-01-01 04:00:00,1,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
summary_stats = pd.DataFrame({
    'Tỷ lệ giờ không mưa (=0) [%]': [
        (df['obs'] == 0).mean() * 100,
        (df['ecmwf_1h'] == 0).mean() * 100,
        (df['gfs_1h'] == 0).mean() * 100,
        (df['mblue_1h'] == 0).mean() * 100,
    ],
    'Tỷ lệ giờ có mưa (>0) [%]': [
        (df['obs'] > 0).mean() * 100,
        (df['ecmwf_1h'] > 0).mean() * 100,
        (df['gfs_1h'] > 0).mean() * 100,
        (df['mblue_1h'] > 0).mean() * 100,
    ],
    'Lượng mưa TB (mm/h)': [
        df['obs'].mean(),
        df['ecmwf_1h'].mean(),
        df['gfs_1h'].mean(),
        df['mblue_1h'].mean(),
    ],
    'Lượng mưa Max (mm/h)': [
        df['obs'].max(),
        df['ecmwf_1h'].max(),
        df['gfs_1h'].max(),
        df['mblue_1h'].max(),
    ]
}, index=['Thực đo (obs)', 'ECMWF', 'GFS', 'Meteoblue'])

display(summary_stats.round(3))

,Tỷ lệ giờ không mưa (=0) [%],Tỷ lệ giờ có mưa (>0) [%],Lượng mưa TB (mm/h),Lượng mưa Max (mm/h)
Thực đo (obs),91.130,8.870,0.210,44.800
ECMWF,62.682,37.318,0.223,10.167
GFS,64.774,35.226,0.193,13.767
Meteoblue,79.061,20.939,0.222,10.033


---
## 3. Kỹ thuật Tạo Đặc trưng Cốt lõi (16 Features)
Đã cắt bỏ 59 đặc trưng dư thừa/đa cộng tuyến (như `sin/cos`, `log1p`, các hiệu từng cặp, tỷ lệ chia, các cửa sổ 6h/12h trùng lặp), giữ lại đúng **16 đặc trưng vật lý cốt lõi**:
1. **Thời gian (2)**: `thang`, `gio`
2. **Dự báo gốc 1h & 24h (6)**: `ecmwf_1h`, `gfs_1h`, `mblue_1h`, `ecmwf_24h`, `gfs_24h`, `mblue_24h`
3. **Thống kê tổ hợp Ensemble 1h (4)**: `ens_1h_mean`, `ens_1h_max`, `ens_1h_std`, `n_models_rain_1h`
4. **Cửa sổ trượt bù lệch pha (4)**: `ecmwf_1h_roll3h_mean`, `gfs_1h_roll3h_mean`, `mblue_1h_roll3h_mean`, `ens_1h_mean_roll6h_max`

In [4]:
def create_features(data: pd.DataFrame) -> pd.DataFrame:
    """Tạo 16 đặc trưng vật lý cốt lõi (không phụ thuộc vào obs quá khứ)."""
    fe = pd.DataFrame(index=data.index)

    # 1. Đặc trưng thời gian (2 biến)
    fe['thang'] = data['thang']
    fe['gio'] = data['gio']

    # 2. Dự báo gốc 1h & 24h (6 biến)
    fe['ecmwf_1h'] = data['ecmwf_1h']
    fe['gfs_1h'] = data['gfs_1h']
    fe['mblue_1h'] = data['mblue_1h']
    fe['ecmwf_24h'] = data['ecmwf_24h']
    fe['gfs_24h'] = data['gfs_24h']
    fe['mblue_24h'] = data['mblue_24h']

    # 3. Thống kê tổ hợp Ensemble 1h (4 biến)
    cols_1h = ['ecmwf_1h', 'gfs_1h', 'mblue_1h']
    fe['ens_1h_mean'] = data[cols_1h].mean(axis=1)
    fe['ens_1h_max'] = data[cols_1h].max(axis=1)
    fe['ens_1h_std'] = data[cols_1h].std(axis=1)
    fe['n_models_rain_1h'] = (data[cols_1h] >= 0.1).sum(axis=1)

    # 4. Cửa sổ trượt bù sai số lệch pha thời gian (4 biến)
    for col in cols_1h:
        fe[f'{col}_roll3h_mean'] = data[col].rolling(window=3, min_periods=1, center=True).mean()
    fe['ens_1h_mean_roll6h_max'] = fe['ens_1h_mean'].rolling(window=6, min_periods=1, center=True).max()

    return fe

# Chia tập Train/Test theo thời gian TRƯỚC khi tạo đặc trưng để tránh rò rỉ cửa sổ trượt
split_idx = int(len(df) * 0.80)
train_raw = df.iloc[:split_idx].copy()
test_raw = df.iloc[split_idx:].copy()

X_train = create_features(train_raw)
X_test = create_features(test_raw)
feature_cols = list(X_train.columns)

y_train = train_raw['obs'].values
y_test = test_raw['obs'].values
test_df = test_raw.copy()

print(f"Tổng số đặc trưng đầu vào ({len(feature_cols)} features): {feature_cols}")
print(f"Tập Huấn luyện (Train): {X_train.shape} | {train_raw['thoigian'].min()} -> {train_raw['thoigian'].max()}")
print(f"Tập Kiểm thử   (Test) : {X_test.shape}  | {test_raw['thoigian'].min()} -> {test_raw['thoigian'].max()}")

Tổng số đặc trưng đầu vào (16 features): ['thang', 'gio', 'ecmwf_1h', 'gfs_1h', 'mblue_1h', 'ecmwf_24h', 'gfs_24h', 'mblue_24h', 'ens_1h_mean', 'ens_1h_max', 'ens_1h_std', 'n_models_rain_1h', 'ecmwf_1h_roll3h_mean', 'gfs_1h_roll3h_mean', 'mblue_1h_roll3h_mean', 'ens_1h_mean_roll6h_max']
Tập Huấn luyện (Train): (17100, 16) | 2024-01-01 00:00:00 -> 2025-12-13 11:00:00
Tập Kiểm thử   (Test) : (4276, 16)  | 2025-12-13 12:00:00 -> 2026-06-09 15:00:00


---
## 4. Huấn luyện các Mô hình Học máy
Huấn luyện trên 16 đặc trưng cốt lõi và lượng tử hóa đầu ra về bậc `0.2 mm`.

In [5]:
def quantize_02mm(arr):
    """Làm tròn lượng mưa về bội số của 0.2mm (độ phân giải gầu đo mưa thực tế)."""
    arr = np.maximum(0.0, np.asarray(arr))
    return np.round(arr / 0.2) * 0.2

# 1. Random Forest Regressor (Huấn luyện trên miền log1p để giảm nhạy cảm với cực trị)
rf_reg = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=10,
    n_jobs=-1,
    random_state=42
)
rf_reg.fit(X_train, np.log1p(y_train))
pred_rf_raw = np.expm1(rf_reg.predict(X_test))
pred_rf = quantize_02mm(pred_rf_raw)

# 2. HistGradientBoosting Regressor (Poisson Loss cho dữ liệu mưa lệch phải)
hgb_poisson = HistGradientBoostingRegressor(
    loss='poisson',
    max_iter=300,
    max_depth=6,
    learning_rate=0.05,
    min_samples_leaf=20,
    l2_regularization=1.0,
    random_state=42
)
hgb_poisson.fit(X_train, y_train)
pred_hgb_poisson = quantize_02mm(hgb_poisson.predict(X_test))

print("Đã huấn luyện xong Random Forest & HGB Poisson!")

Đã huấn luyện xong Random Forest & HGB Poisson!


In [6]:
# 3. Mô hình 2 Pha (Two-Stage Hurdle Model)
# Chia tập Train thành Sub-Train (85%) và Validation (15%) để tìm ngưỡng xác suất mưa tối ưu
val_split = int(len(X_train) * 0.85)
X_subtrain, X_val = X_train.iloc[:val_split], X_train.iloc[val_split:]
y_subtrain, y_val = y_train[:val_split], y_train[val_split:]

# Pha 1: Phân loại Có mưa (>= 0.2mm) hay Không mưa
clf_rain = HistGradientBoostingClassifier(
    max_iter=300,
    max_depth=5,
    learning_rate=0.05,
    min_samples_leaf=30,
    l2_regularization=2.0,
    random_state=42
)
clf_rain.fit(X_subtrain, (y_subtrain >= 0.2).astype(int))

# Tìm ngưỡng xác suất tối ưu F1-Score trên tập Validation
val_probs = clf_rain.predict_proba(X_val)[:, 1]
best_thresh, best_f1 = 0.5, 0.0
for th in np.arange(0.15, 0.70, 0.02):
    score = f1_score((y_val >= 0.2).astype(int), (val_probs >= th).astype(int))
    if score > best_f1:
        best_f1 = score
        best_thresh = th

# Huấn luyện lại bộ phân loại trên toàn bộ tập Train
clf_rain.fit(X_train, (y_train >= 0.2).astype(int))

# Pha 2: Hồi quy cường độ mưa (chỉ học trên các giờ thực tế có mưa >= 0.2mm)
rain_mask_train = y_train >= 0.2
reg_hurdle = HistGradientBoostingRegressor(
    loss='squared_error',
    max_iter=300,
    max_depth=5,
    learning_rate=0.05,
    min_samples_leaf=15,
    l2_regularization=2.0,
    random_state=42
)
reg_hurdle.fit(X_train[rain_mask_train], np.log1p(y_train[rain_mask_train]))

# Kết hợp Pha 1 & Pha 2 trên tập Test
test_rain_prob = clf_rain.predict_proba(X_test)[:, 1]
test_rain_amount = np.expm1(reg_hurdle.predict(X_test))
pred_hurdle = np.where(test_rain_prob >= best_thresh, np.maximum(0.2, quantize_02mm(test_rain_amount)), 0.0)

print(f"Ngưỡng xác suất tối ưu (Two-Stage Hurdle): {best_thresh:.2f}")

Ngưỡng xác suất tối ưu (Two-Stage Hurdle): 0.15


---
## 5. Đánh giá và So sánh Kết quả trên Tập Kiểm thử (Test Set)

In [7]:
def evaluate_predictions(y_true, y_pred, model_name, rain_threshold=0.2):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    bin_true = (y_true >= rain_threshold).astype(int)
    bin_pred = (y_pred >= rain_threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(bin_true, bin_pred).ravel()
    acc = accuracy_score(bin_true, bin_pred) * 100
    pod = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0.0
    far = fp / (tp + fp) * 100 if (tp + fp) > 0 else 0.0
    csi = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
    f1 = f1_score(bin_true, bin_pred)

    return {
        'Mô hình': model_name,
        'MAE (mm)': round(mae, 4),
        'RMSE (mm)': round(rmse, 4),
        'Accuracy Mưa/Không mưa (%)': round(acc, 2),
        'POD / Recall (%)': round(pod, 2),
        'FAR - Báo động giả (%)': round(far, 2),
        'CSI (Threat Score)': round(csi, 4),
        'F1-Score (Có mưa)': round(f1, 4)
    }

results_1h = [
    evaluate_predictions(y_test, test_df['ecmwf_1h'].values, '1. Dự báo gốc ECMWF'),
    evaluate_predictions(y_test, test_df['gfs_1h'].values, '2. Dự báo gốc GFS'),
    evaluate_predictions(y_test, test_df['mblue_1h'].values, '3. Dự báo gốc Meteoblue'),
    evaluate_predictions(y_test, X_test['ens_1h_mean'].values, '4. Trung bình cộng 3 mô hình'),
    evaluate_predictions(y_test, pred_rf, '5. ML - Random Forest'),
    evaluate_predictions(y_test, pred_hgb_poisson, '6. ML - HGB Poisson'),
    evaluate_predictions(y_test, pred_hurdle, '7. ML - Two-Stage Hurdle GBDT'),
]

df_eval_1h = pd.DataFrame(results_1h)
display(df_eval_1h)

,Mô hình,MAE (mm),RMSE (mm),Accuracy Mưa/Không mưa (%),POD / Recall (%),FAR - Báo động giả (%),CSI (Threat Score),F1-Score (Có mưa)
0,1. Dự báo gốc ECMWF,0.2330,0.9163,81.01,53.96,82.67,0.1501,0.2610
1,2. Dự báo gốc GFS,0.2008,0.8942,84.71,43.02,81.52,0.1486,0.2588
2,3. Dự báo gốc Meteoblue,0.2400,1.0110,84.10,45.28,81.68,0.1500,0.2609
3,4. Trung bình cộng 3 mô hình,0.2173,0.8884,83.82,52.08,79.79,0.1667,0.2857
4,5. ML - Random Forest,0.1460,0.8920,91.11,31.70,71.91,0.1721,0.2937
5,6. ML - HGB Poisson,0.1674,0.8919,88.91,39.62,75.97,0.1759,0.2991
6,7. ML - Two-Stage Hurdle GBDT,0.1571,0.9277,90.29,41.13,70.38,0.2080,0.3444


In [8]:
def phan_cap_mua_24h(r24_series):
    r = np.asarray(r24_series)
    cats = np.zeros(len(r), dtype=int)
    cats[(r > 0.0) & (r <= 0.6)] = 1
    cats[(r > 0.6) & (r <= 6.0)] = 2
    cats[(r > 6.0) & (r <= 16.0)] = 3
    cats[(r > 16.0) & (r <= 50.0)] = 4
    cats[(r > 50.0) & (r <= 100.0)] = 5
    cats[r > 100.0] = 6
    return cats

def calc_forward_24h(arr):
    s = pd.Series(arr)
    return s.iloc[::-1].rolling(window=24, min_periods=1).sum().iloc[::-1].values

obs_24h_test = calc_forward_24h(y_test)
obs_phancap_test = phan_cap_mua_24h(obs_24h_test)

models_24h_eval = {
    '1. Dự báo gốc ECMWF': phan_cap_mua_24h(test_df['ecmwf_24h'].values),
    '2. Dự báo gốc GFS': phan_cap_mua_24h(test_df['gfs_24h'].values),
    '3. Dự báo gốc Meteoblue': phan_cap_mua_24h(test_df['mblue_24h'].values),
    '5. ML - Random Forest': phan_cap_mua_24h(calc_forward_24h(pred_rf)),
    '6. ML - HGB Poisson': phan_cap_mua_24h(calc_forward_24h(pred_hgb_poisson)),
    '7. ML - Two-Stage Hurdle GBDT': phan_cap_mua_24h(calc_forward_24h(pred_hurdle)),
}

rows_24h = []
for name, pred_pc in models_24h_eval.items():
    dung = int(np.sum(pred_pc == obs_phancap_test))
    sai = int(np.sum(pred_pc != obs_phancap_test))
    ty_le_dung = dung / len(obs_phancap_test) * 100
    ty_le_sai_lech_1cap = np.mean(np.abs(pred_pc - obs_phancap_test) <= 1) * 100
    rows_24h.append({
        'Mô hình': name,
        'Số mẫu Đúng': dung,
        'Số mẫu Sai': sai,
        'Tỷ lệ Đúng phân cấp 24h (%)': round(ty_le_dung, 2),
        'Tỷ lệ Đúng trong phạm vi ±1 cấp (%)': round(ty_le_sai_lech_1cap, 2)
    })

df_eval_24h = pd.DataFrame(rows_24h)
display(df_eval_24h)

,Mô hình,Số mẫu Đúng,Số mẫu Sai,Tỷ lệ Đúng phân cấp 24h (%),Tỷ lệ Đúng trong phạm vi ±1 cấp (%)
0,1. Dự báo gốc ECMWF,1489,2787,34.82,66.98
1,2. Dự báo gốc GFS,2147,2129,50.21,76.19
2,3. Dự báo gốc Meteoblue,2447,1829,57.23,74.95
3,5. ML - Random Forest,2402,1874,56.17,79.49
4,6. ML - HGB Poisson,2255,2021,52.74,75.75
5,7. ML - Two-Stage Hurdle GBDT,2764,1512,64.64,76.40


---
## 6. Trực quan hóa Kết quả

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(
    data=df_eval_24h,
    y='Mô hình',
    x='Tỷ lệ Đúng phân cấp 24h (%)',
    palette='viridis',
    ax=axes[0]
)
axes[0].set_title('So sánh Tỷ lệ Dự báo Đúng Phân cấp mưa 24h (%) trên tập Test')
axes[0].set_xlim(0, 100)

sample_slice = slice(1500, 2000)
axes[1].plot(test_df['thoigian'].iloc[sample_slice], obs_24h_test[sample_slice], label='Thực đo (obs 24h)', color='black', linewidth=2)
axes[1].plot(test_df['thoigian'].iloc[sample_slice], test_df['ecmwf_24h'].iloc[sample_slice], label='ECMWF 24h', alpha=0.6, linestyle='--')
axes[1].plot(test_df['thoigian'].iloc[sample_slice], calc_forward_24h(pred_hurdle)[sample_slice], label='ML Hurdle 24h (Hiệu chỉnh)', color='red', linewidth=1.8)
axes[1].set_title('So sánh Lượng mưa tích lũy 24h (Mẫu 500 giờ trên tập Test)')
axes[1].set_ylabel('Lượng mưa 24h (mm)')
axes[1].legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

---
## 7. Lưu Mô hình và Hàm Dự báo Thực tế cho "10:00 Ngày Mai" (Trên Chuỗi Liên tục)
**Lưu ý quan trọng**: Hàm `du_bao_mua_hieu_chinh` nhận vào chuỗi dự báo **liên tục theo giờ** để tính đúng cửa sổ trượt `roll3h_mean` và `roll6h_max`, sau đó mới trích xuất khung giờ mục tiêu (ví dụ `gio == 10`).

In [10]:
model_bundle = {
    'clf_rain': clf_rain,
    'reg_hurdle': reg_hurdle,
    'best_thresh': best_thresh,
    'feature_cols': feature_cols
}
joblib.dump(model_bundle, MODELS_DIR / 'mo_hinh_hieu_chinh_mua_banho.joblib')
print("Đã lưu mô hình vào: mo_hinh_hieu_chinh_mua_banho.joblib")

def du_bao_mua_hieu_chinh(df_chuoi_lien_tuc: pd.DataFrame, model_path=MODELS_DIR / 'mo_hinh_hieu_chinh_mua_banho.joblib') -> pd.DataFrame:
    """
    Nhận vào DataFrame chuỗi dự báo LIÊN TỤC THEO GIỜ của ECMWF, GFS, Meteoblue
    (gồm thoigian, thang, gio, ecmwf_1h, gfs_1h, mblue_1h, ecmwf_24h, gfs_24h, mblue_24h)
    và trả về lượng mưa hiệu chỉnh 1h bậc 0.2mm cho từng giờ.
    """
    bundle = joblib.load(model_path)
    fe_new = create_features(df_chuoi_lien_tuc)
    X_new = fe_new[bundle['feature_cols']]

    prob_rain = bundle['clf_rain'].predict_proba(X_new)[:, 1]
    raw_amount = np.expm1(bundle['reg_hurdle'].predict(X_new))
    pred_obs_1h = np.where(prob_rain >= bundle['best_thresh'], np.maximum(0.2, quantize_02mm(raw_amount)), 0.0)

    ket_qua = df_chuoi_lien_tuc[['thoigian', 'ecmwf_1h', 'gfs_1h', 'mblue_1h']].copy()
    ket_qua['xac_suat_mua_ML (%)'] = (prob_rain * 100).round(1)
    ket_qua['du_bao_mua_ML_1h (mm)'] = np.round(pred_obs_1h, 2)
    return ket_qua

# Tính dự báo trên chuỗi liên tục trước, sau đó mới lọc lấy khung giờ 10:00
ket_qua_test_full = du_bao_mua_hieu_chinh(df.iloc[split_idx:].copy())
ket_qua_test_full['obs_thuc_do (mm)'] = df.iloc[split_idx:]['obs'].values
ket_qua_10h = ket_qua_test_full[df.iloc[split_idx:]['gio'] == 10].tail(10)
display(ket_qua_10h)

Đã lưu mô hình vào: mo_hinh_hieu_chinh_mua_banho.joblib


,thoigian,ecmwf_1h,gfs_1h,mblue_1h,xac_suat_mua_ML (%),du_bao_mua_ML_1h (mm),obs_thuc_do (mm)
21154,2026-05-31 10:00:00,0.0667,0.0000,0.0000,4.2,0.0,0.0
21178,2026-06-01 10:00:00,0.0000,0.0000,0.0000,8.2,0.0,0.0
21202,2026-06-02 10:00:00,0.0000,0.0667,0.0000,4.4,0.0,0.0
21226,2026-06-03 10:00:00,0.0667,0.0000,0.0000,3.5,0.0,0.0
21250,2026-06-04 10:00:00,0.0000,0.3333,0.5000,20.3,0.2,0.0
21274,2026-06-05 10:00:00,0.0000,0.5000,0.0000,14.8,0.0,0.0
21298,2026-06-06 10:00:00,0.0000,0.0000,0.0000,7.5,0.0,0.0
21322,2026-06-07 10:00:00,0.1000,0.0000,0.0000,4.6,0.0,0.0
21346,2026-06-08 10:00:00,0.0667,0.0000,0.0000,11.6,0.0,0.0
21370,2026-06-09 10:00:00,1.2000,0.1333,1.3333,16.2,0.2,8.2
